# NB 07 · Synthesis

**Series:** Learning LangGraph through literate notebooks, notebook 7 of 7.
**Central question:** what are the answers to the four research questions, and how far do they reach?

Six notebooks built one research assistant, one concept at a time. This notebook steps back. It
answers each research question with the evidence the series produced, names the limits of that
evidence, and sketches the next step that the project deliberately did not take: several agents
behind a supervisor.

## Learning goals

After this notebook you can:

- answer each of the four research questions with evidence from a specific notebook;
- explain where a linear chain becomes structurally insufficient, using the assistant's own runs;
- describe how the state contract kept a shared state consistent as the graph grew;
- weigh rule-based checks against a model's own judgement, including the limits of both;
- sketch how the assistant's nodes would become agents behind a supervisor.

## Setup

This notebook synthesises **recorded** evidence. It therefore always uses the scripted model, which
replays the replies recorded from `qwen3:8b` for NB 01 to NB 06, whatever `LLM_MODE` is set to: a
synthesis must cite the same evidence on every run. How live replies vary is documented in NB 04 and
NB 05.

In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

import nbformat
from dotenv import load_dotenv
from langgraph.graph import END, START, StateGraph

from research_assistant import assistant, judgement, loop, persistence, skeleton
from research_assistant.assistant import collect_primary, collect_secondary, make_score, write_answer
from research_assistant.evidence import CANONICAL_QUESTIONS, assess
from research_assistant.models import get_chat_model
from research_assistant.retrieval import retrieve
from research_assistant.state import ResearchState

load_dotenv()

assert sys.version_info >= (3, 12), "This series needs Python 3.12 or newer."
print(f"Python {sys.version.split()[0]}")
for package in ("langgraph", "langchain-core"):
    print(f"{package} {version(package)}")

RECORDINGS = ("nb01_hello.yaml", "nb03_claims.yaml", "nb04_queries.yaml", "nb05_judgements.yaml", "nb06_constructed.yaml")
llm = get_chat_model("fake", fixture=RECORDINGS)
QUESTIONS = CANONICAL_QUESTIONS
NOTEBOOKS = sorted(Path(".").glob("0[1-6]_*.ipynb"))
print(f"Recorded replies: {len(llm.rules)} | notebooks analysed: {len(NOTEBOOKS)}")

## Concept: answers with evidence

Each research question gets the same treatment: the answer in a few sentences, the evidence with the
notebook it comes from, and the limits of that evidence. Where a number can be computed from the
notebooks or the package, the cell below computes it rather than quoting it from memory.

| # | Research question (short form) | Main evidence |
|---|---|---|
| 1 | Where does a linear chain become structurally insufficient? | NB 01, NB 04 |
| 2 | How does a shared state stay consistent, and what does the state contract contribute? | NB 02, NB 04, NB 06 |
| 3 | How can concepts, code and mistakes, including model misjudgements, form one thread? | all notebooks |
| 4 | How can evidence quality be assessed, and where do rule-based heuristics reach their limits? | NB 03, NB 05 |

## Experiment

### 1 Research question 1: graph against chain

> *How does LangGraph's graph-based control flow differ from linear chain approaches, and at which
> concrete point in the research process does a linear chain become structurally inadequate?*

**Answer.** For a single straight step, there is no difference in outcome: NB 01 built the same step
as an LCEL chain and as a graph, and both gave the same answer. The difference appears at the quality
gate. When the evidence is insufficient, the right next step is not "write the answer" but "search
again, differently". A chain runs its steps once, in a fixed order; it has no way back. A graph needs
one conditional edge from the router to the search, and a state that carries what has been found.

**Evidence.** The cell runs the same nodes twice for topics c and e: once as the straight line of
NB 03, once with the router and loop of NB 04.

In [ ]:
straight = assistant.build_research_graph(llm)
with_loop = loop.build_loop_graph(llm)
for topic in ("e", "c"):
    print(f"topic {topic}")
    print("  straight line:", straight.invoke({"question": QUESTIONS[topic]})["draft_answer"][:95], "...")
    print("  with the loop:", with_loop.invoke({"question": QUESTIONS[topic]})["draft_answer"][:95], "...")

Topic e is where the chain becomes inadequate in the most concrete sense: the evidence that answers
the question exists (E4), but only a second, differently worded search finds it. The straight line
stops at "evidence insufficient"; the loop finds E4 in round 2 and passes the gate. Topic c shows the
other half: a loop must also know when to stop. After three rounds, the graph ends with a stated
limitation instead of searching forever (NB 04, mistake 1).

| Capability | LCEL chain | LangGraph graph |
|---|---|---|
| a fixed sequence of steps | yes | yes |
| steps that run in parallel | limited | yes (NB 02, NB 03) |
| a next step chosen from the state | no | yes, with a router (NB 04) |
| going back to an earlier step | no | yes, with an edge back (NB 04) |
| a shared, typed state across steps | no, output is passed forward | yes, with reducers (NB 02) |
| pausing for a person and resuming later | no | yes, with checkpoints (NB 06) |

**Limits.** The comparison uses LangChain's expression language as the chain approach; LangChain can
build loops in plain Python around a chain, at the cost of managing the state by hand. The claim is
about what each framework expresses structurally, not about what is possible with enough extra code.

### 2 Research question 2: one consistent state

> *How can a shared state be kept consistent across several coordinated processing steps, and what
> does an explicit state contract with defined field ownership contribute?*

**Answer.** Three mechanisms worked together. **Reducers** decide how writes to the same field
combine; without one, LangGraph either loses data silently (two writes in sequence) or refuses (two at
once). **Ownership** says which node may write which field; LangGraph does not enforce it, so the
contract was written down as data and checked on every run. **Invariants** check that the state still
supports the decisions taken on it; NB 04 found the case where the router chose correctly and the
state, without a merge rule for claims, contradicted that choice afterwards.

**Evidence.** The contract grew with the graph, and every new field arrived with its owner:

In [ ]:
contracts = {
    "NB 02": skeleton.SKELETON_CONTRACT,
    "NB 03": assistant.RESEARCH_CONTRACT,
    "NB 04": loop.LOOP_CONTRACT,
    "NB 06": persistence.REVIEW_CONTRACT,
}
fields = list(dict.fromkeys(field for contract in contracts.values() for field in contract))
for field in fields:
    previous, label = None, field
    for name, contract in contracts.items():
        owners = ", ".join(sorted(contract.get(field, set())))
        if owners and owners != previous:
            print(f"{label:<20}{name}: {owners.replace('*', 'every node')}")
            previous, label = owners, ""

| Failure | Where | What it showed |
|---|---|---|
| sequential writes without a reducer | NB 02, experiment 2 | data lost with no error at all |
| parallel writes without a reducer | NB 02, mistake 2 | LangGraph refuses: `InvalidUpdateError` |
| a node writing a field it does not own | NB 02, mistake 3 | allowed by LangGraph, caught by the contract check |
| a field written every round, without a merge rule | NB 04, mistake 3 | the state contradicts the router's decision |
| a paused run resumed on the wrong thread | NB 06, mistake 2 | the decision is lost; the error names a missing question |

**Limits.** The contract is a convention checked after the fact on the updates a run reports; it does
not stop a write while it happens, and it is not a security boundary (ADR 0005). The research question
speaks of "processing steps (nodes)", which is how the lecturer's concept frames it; section 6 sketches
the original wording's "several cooperating agents".

### 3 Research question 3: one thread through concepts, code and mistakes

> *How can concepts, code and typical mistakes, including deliberately provoked misjudgements by the
> LLM itself, be woven together so that a common thread runs through the notebook series?*

**Answer.** Three devices carried the thread. **One system** grew through all six notebooks, so every
concept answered a weakness the previous notebook had made visible. **One fixed structure** per
notebook (eight sections, the same order every time) made the notebooks comparable. **Checked
mistakes** turned errors into specifications: every deliberate mistake is either declared with its
exact exception type, which the test runner enforces, or checked by its effect when it fails silently.

In [ ]:
print(f"{'notebook':<42}{'mistakes':>9}{'raising':>9}{'silent':>8}  exception types")
totals = [0, 0]
for path in NOTEBOOKS:
    notebook = nbformat.read(path, as_version=4)
    raising = [c.metadata["expected_exception"] for c in notebook.cells if "raises-exception" in c.metadata.get("tags", [])]
    mistakes = sum(1 for c in notebook.cells if c.cell_type == "markdown" and "### Mistake" in c.source)
    totals[0] += len(raising); totals[1] += mistakes - len(raising)
    print(f"{path.name:<42}{mistakes:>9}{len(raising):>9}{mistakes - len(raising):>8}  {', '.join(raising) or '-'}")
print(f"{'total':<42}{sum(totals):>9}{totals[0]:>9}{totals[1]:>8}")

Two choices kept the model's part of the thread verifiable. Every scripted reply was **recorded** from
`qwen3:8b`, never invented, so the model's misjudgements in NB 05 are its own. And every result that
contradicted the plan was reported as found: the guessed relevance threshold that failed (NB 03), the
recency error that did not occur (NB 05), the persistence errors that appeared in a different place
than expected (NB 06).

**Limits.** Whether the thread works for a reader is a didactic claim that the tests cannot check;
only feedback from readers can answer it.

### 4 Research question 4: evidence quality, rules and the model

> *How can the quality of evidence be assessed in an LLM-supported research process, and where are the
> limits of rule-based heuristics compared with cases in which the LLM itself overlooks
> contradictions or weights sources wrongly?*

**Answer.** By splitting the work: the model reads one document at a time and reports one figure;
fixed rules judge (authority floor, independence, recency, contradictions, quality gate, ADR 0007).
Asked to judge itself, the model agreed with the rules completely only on the control case b. On topic d
it was partly right (recency judged correctly, the resolved contradiction reported as none); on the
other three cases it disagreed on substance:

In [ ]:
by_id = {doc["source_id"]: doc for doc in assistant.corpus()}
cases = {topic: retrieve(QUESTIONS[topic], assistant.corpus(), assistant.weights()) for topic in "abde"}
cases["e, copies only"] = [by_id["E1"], by_id["E2"]]
for name, documents in cases.items():
    question = QUESTIONS[name[0]]
    claims = {doc["source_id"]: assistant.extract_claim(llm, question, doc) for doc in documents}
    verdict = judgement.judge(llm, question, documents)
    differs = [field for field, _, _, agree in judgement.compare(verdict, assess(documents, claims), documents, claims) if not agree]
    print(f"{name:<15} model differs from the rules on: {', '.join(differs) or 'nothing'}")

The rules have limits of their own, and the series met each of them:

- **Hand-assigned authority weights.** Every weight is a judgement made when the corpus was written;
  change one, and an outcome changes (NB 03, section 8; NB 05, exercise).
- **A measured threshold with narrow margins.** The relevance rule separates all eight measured queries,
  but by 0.04 and 0.06 at the closest points (ADR 0007). A new document could cross the line.
- **Word matching, not meaning.** The guessed threshold failed because "third quarter" is as rare as
  "Solvane" in 21 documents; the fix is a heuristic about capitalised names (NB 03).
- **Rules only see what they were built to see.** The rules detect a contradiction between figures; a
  contradiction expressed in words, or a copy without a `based_on` entry, would pass unnoticed.

The model's limits were different in kind: they shifted. A reworded prompt fixed one case and broke
another; metadata in view changed nothing; thinking mode fixed one case at six times the cost (NB 05);
at temperature 0, the same prompt still produced different wording (NB 04). Rule-based checks fail the
same way every time and can be inspected; that, more than their accuracy, is why the assistant lets
them judge.

**Limits.** One model, one corpus of 21 documents, two prompt versions. The result is a documented case,
not a general law about language models.

### 5 Limits of the whole approach

- **Small, fictional corpus.** It makes every case controllable and every result reproducible; it also
  means nothing here measures performance on real documents.
- **Retrieval by words.** Production systems would use semantic search; the series chose explainable
  word matching on purpose (ADR 0007).
- **One local model.** `qwen3:8b` with temperature 0. Larger or differently trained models may judge
  differently; the comparison method of NB 05 is reusable for testing them.
- **A simulated reviewer.** NB 06 shows the mechanics of human review; it does not test how real
  reviewers decide, or how long a review may wait.

### 6 Outlook: from coordinated nodes to cooperating agents

The project deliberately built one graph with coordinated nodes, not a multi-agent system (ADR 0005):
within 50 to 60 hours, coordination logic, emergent behaviour and a contract for several actors would
not have fitted. The step is nonetheless small in structure, because the nodes already have clear
responsibilities. Grouped into three agents behind a supervisor:

```text
                     +---------------+
                     |  Supervisor   |  decides which agent works next
                     +---------------+
                     /       |        \
       +--------------+ +-----------+ +---------------+
       |  Retrieval   | |  Scoring  | |   Synthesis   |
       |  agent       | |  agent    | |   agent       |
       +--------------+ +-----------+ +---------------+
        collectors,       claims and     answer, or
        query rewriting   evidence       limitation and
                          rules          review
```

The state contract stays the coordination mechanism: each agent owns the fields it writes. In LangGraph,
each agent can be a compiled **subgraph** used as a node of the parent graph. The cell below is a sketch
of that structure, built from the nodes that already exist, with a deterministic supervisor; it is not
part of the assistant.

In [ ]:
def agent(*nodes):
    builder = StateGraph(ResearchState)
    for name, node in nodes:
        builder.add_node(name, node)
        builder.add_edge(START, name)
        builder.add_edge(name, END)
    return builder.compile()


retrieval_agent = agent(("collect_primary", collect_primary), ("collect_secondary", collect_secondary))
scoring_agent = agent(("score", make_score(llm)))
synthesis_agent = agent(("write_answer", write_answer))


def supervisor(state: ResearchState) -> dict:
    return {}


def next_agent(state: ResearchState) -> str:
    if not state.get("documents"):
        return "retrieval"
    if "evidence_score" not in state:
        return "scoring"
    if "draft_answer" not in state:
        return "synthesis"
    return "done"


team = StateGraph(ResearchState)
team.add_node("supervisor", supervisor)
team.add_node("retrieval", retrieval_agent)
team.add_node("scoring", scoring_agent)
team.add_node("synthesis", synthesis_agent)
team.add_edge(START, "supervisor")
team.add_conditional_edges("supervisor", next_agent,
                           {"retrieval": "retrieval", "scoring": "scoring", "synthesis": "synthesis", "done": END})
for name in ("retrieval", "scoring", "synthesis"):
    team.add_edge(name, "supervisor")
team_graph = team.compile()

print(team_graph.invoke({"question": QUESTIONS["a"]})["draft_answer"][:90], "...")

The three agents give the same answer as the single graph of NB 03, which is the point of the sketch:
the decomposition is already in the design. What a real multi-agent version would add, and what the
estimated 20 to 30 extra hours would go into:

- each agent with its own prompts, and possibly its own model;
- private working state per agent, with only agreed fields shared (ADR 0005 notes that such separation
  organises data flow and is not a security boundary);
- a supervisor that may itself be a model, which brings back the question of NB 05: can its routing
  decisions be checked as well as the router's rules?
- a contract for several actors, including which agent may overrule which.

## Common mistakes

The mistakes of a synthesis are mistakes of reasoning. Two are easy to make with the evidence above.

### Mistake 1: generalising from one recorded failure

From NB 05 one might write: *"the model fails to detect contradictions"*. The recorded evidence does not
support that sentence. The cell checks it against the recordings.

In [ ]:
documents_e = retrieve(QUESTIONS["e"], assistant.corpus(), assistant.weights())
listing_verdict = judgement.judge(llm, QUESTIONS["e"], documents_e, prompt=judgement.LISTING_JUDGE_PROMPT)
control_verdict = judgement.judge(llm, QUESTIONS["b"], retrieve(QUESTIONS["b"], assistant.corpus(), assistant.weights()))
print("version 2 prompt, topic e: contradiction detected =", listing_verdict.contradiction)
print("control case b: model says no contradiction, rules agree =", control_verdict.contradiction is False)
assert listing_verdict.contradiction is True, "the claim 'it never detects contradictions' is refuted by the recordings"

With a reworded prompt, the same model did detect the contradiction in topic e, and on the control
case it was right to see none. The supported statement is narrower: *with the first prompt version, the
model missed the contradictions in topics a and e*. **Fix:** state findings with the model, the prompt
and the cases they hold for, as NB 05 does.

### Mistake 2: reading passing tests as proof that the model is reliable

All tests pass. But the tests replay recorded replies; they show that the code handles *those* replies
correctly, not that the model will give them again, or that it will answer anything else. A question
nobody recorded shows the boundary at once:

In [ ]:
assistant.extract_claim(llm, "What was Solvane Dynamics' revenue in the second quarter of 2025?", by_id["A1"])

*"no fixture rule matches prompt"*. The scripted model refuses rather than improvise, which is the
right behaviour for a test, and exactly why test results say nothing about live behaviour. **Fix:**
keep the two kinds of evidence apart. Recorded replies make the code's behaviour reproducible; live
runs, reported as such (NB 04, NB 05), show how the model actually behaves.

## Prompt log

This notebook adds no prompt: it synthesises recorded evidence and replays the prompts of NB 01 to
NB 06. The prompt logs of NB 01, NB 03, NB 04 and NB 05 hold the iterations; NB 02 and NB 06 explain why
they needed none.

## Summary

**Research question 1:** a chain is enough for a fixed sequence; it becomes inadequate at the quality
gate, where the next step depends on what was found. Topic e passes the gate only because the graph can
search again, and topic c ends responsibly only because the loop has a stop condition.

**Research question 2:** reducers, ownership and invariants keep the shared state consistent; the
contract makes ownership visible and checkable, and every failure without it was silent or misleading.

**Research question 3:** one growing system, one fixed structure and checked mistakes (18 across six
notebooks: 12 with an exact exception, 6 checked by their effect) carry the thread; recorded, never
invented, model replies keep it verifiable.

**Research question 4:** let the model read and the rules judge. The rules have limits, which the series
names; the model's judgement shifted with prompt, setting and call, which is the stronger reason to keep
judging in rules that can be inspected.

## Exercise

1. Choose one research question and argue the opposite of the answer above, using only evidence from the
   notebooks. Where is the evidence for the stated answer weakest?
2. Extend the supervisor sketch so that the scoring agent can send the work back to the retrieval agent,
   as the router of NB 04 does. Which field does the supervisor need to read, and which agent should own
   the round counter?